# Drone Radar Detection & Classification System (77 GHz FMCW)

This system processes millimeter-wave radar data from the **Zenodo SAAB SIRS 77 GHz FMCW dataset (Record 5896641)** to classify **Drones, Birds, and Humans**.

### Project Moat (Competitive Advantage):
1. **Zero-Shot / Leave-One-Drone-Out (LODO) Generalization**: The model learns invariant rotor blade kinematics, allowing it to correctly classify unseen drone airframes never encountered in training.
2. **Dual-Channel Doppler Fusion**: Fuses 2D Range-Doppler spatial profiles with Micro-Doppler Time-Frequency spectrograms.
3. **Ultra-Lightweight Edge Deployment**: Designed with Doppler-separable convolutions (<150k parameters, <1 MB size, <5 ms inference per frame).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add src to Python path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config
from src.preprocess import raw_to_dual_channel, parse_target_labels
from src.model import get_model
from src.dataset import load_dataset_splits

print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Raw Data Path:", config.RAW_DATA_PATH.exists())

## 1. Exploratory Data Analysis & Doppler Visualizations
Let's compare the micro-Doppler signatures of a **Drone (fast rotor blade flashes)** vs a **Bird (slow flapping motion)** vs a **Human (arm/leg pendulum dynamics)**.

In [ ]:
data_path = config.PROCESSED_DIR / "radar_balanced_dataset.npz"
processed_data = np.load(data_path, allow_pickle=True)

X = processed_data["X"]
y = processed_data["y_coarse"]
drone_models = processed_data["drone_models"]

print(f"Total Preprocessed Samples: {len(X):,}")
print(f"Feature Tensor Shape: {X.shape} (Channels=2: Range-Doppler & Micro-Doppler STFT)")

# Find representative samples for Drone, Bird, and Human
sample_indices = {}
for cls_name, cls_idx in config.COARSE_LABEL_TO_IDX.items():
    matches = np.where(y == cls_idx)[0]
    if len(matches) > 0:
        sample_indices[cls_name.capitalize()] = matches[15]

# Plot Dual-Channel Signatures
fig, axes = plt.subplots(len(sample_indices), 2, figsize=(10, 3.5 * len(sample_indices)))

for row, (cls_name, idx) in enumerate(sample_indices.items()):
    rd_channel = X[idx, 0]    # Range-Doppler Map
    stft_channel = X[idx, 1]  # Micro-Doppler STFT
    
    # Range-Doppler Map
    im0 = axes[row, 0].imshow(rd_channel, aspect='auto', cmap='magma', origin='lower')
    axes[row, 0].set_title(f"{cls_name} - Range-Doppler Map", fontsize=11, fontweight='bold')
    axes[row, 0].set_xlabel("Velocity / Doppler Bins")
    axes[row, 0].set_ylabel("Range Bins")
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.04)
    
    # Micro-Doppler STFT
    im1 = axes[row, 1].imshow(stft_channel, aspect='auto', cmap='viridis', origin='lower')
    axes[row, 1].set_title(f"{cls_name} - Micro-Doppler Spectrogram", fontsize=11, fontweight='bold')
    axes[row, 1].set_xlabel("Dwell Time Slices")
    axes[row, 1].set_ylabel("Doppler Frequency")
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 2. Train AeroRadarNet with Leave-One-Drone-Out (LODO)
We train the model while withholding one drone model (e.g. `D6`) from the training set. This verifies whether the model has learned true kinematic Doppler invariants.

In [ ]:
from src.train import run_training

# Run training for 8 epochs (runs quickly on CPU)
model_path, test_loader, meta = run_training(
    mode="lodo",
    holdout_drone="D6",
    epochs=8,
    batch_size=64,
    lr=1e-3
)

## 3. Evaluation & Moat Verification
Evaluate on the test set and display the confusion matrix and zero-shot accuracy on the unseen drone model.

In [ ]:
from src.evaluate import run_evaluation

results = run_evaluation(model_path=model_path, mode="lodo", holdout_drone="D6")

print("\n--- KEY MOAT RESULT ---")
print(f"Zero-Shot Generalization on Unseen Drone D6: {results['unseen_drone_accuracy']*100:.2f}%")
print(f"Inference Latency: {results['latency_ms']:.2f} ms ({results['fps']:.1f} FPS)")

## 4. Single-Sweep Real-Time Prediction
Test the predictor on any arbitrary sample to inspect class confidence scores.

In [ ]:
from src.predict import RadarPredictor

predictor = RadarPredictor(model_path=model_path)

# Select a test sample
test_idx = 100
sample = X[test_idx]
true_label = config.COARSE_CLASSES[int(y[test_idx])]

prediction = predictor.predict_features(sample)

print(f"Actual Target:     {true_label}")
print(f"Predicted Target:  {prediction['prediction']}")
print(f"Confidence:        {prediction['confidence']*100:.2f}%")
print("Probabilities:", prediction['probabilities'])